In [0]:
# Célula 1: cria o schema Gold
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

In [0]:
# Célula 2: mostra as colunas disponíveis em tb_info_filmes
spark.table("workspace.silver.tb_info_filmes").printSchema()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_silver_filmes = spark.table("workspace.silver.tb_info_filmes")

# Window sem partição: só serve para gerar uma numeração sequencial única (a chave substituta)
window_spec = Window.orderBy("id_filme")

df_gold_dim_filmes = df_silver_filmes.select(
    "id_filme",
    "titulo",
    "titulo_original",
    "data_lancamento",
    "ano_lancamento",
    F.col("duracao_minutos").try_cast("int").alias("duracao_minutos"),
    "idioma_original",
    "status_filme",
    "sinopse",
    "frase_divulgacao"
).withColumn("sk_filme", F.row_number().over(window_spec))

# Reordena para deixar a chave substituta como primeira coluna
df_gold_dim_filmes = df_gold_dim_filmes.select(
    "sk_filme", "id_filme", "titulo", "titulo_original", "data_lancamento",
    "ano_lancamento", "duracao_minutos", "idioma_original", "status_filme",
    "sinopse", "frase_divulgacao"
)

df_gold_dim_filmes.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.dim_filmes")

print(f"Tabela gold.dim_filmes gravada com {df_gold_dim_filmes.count()} linhas.")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_silver_generos = spark.table("workspace.silver.tb_generos")

window_spec_genero = Window.orderBy("nome_genero")

df_gold_dim_generos = df_silver_generos.select("nome_genero").distinct() \
    .withColumn("sk_genero", F.row_number().over(window_spec_genero)) \
    .select("sk_genero", "nome_genero")

df_gold_dim_generos.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.dim_generos")

print(f"Tabela gold.dim_generos gravada com {df_gold_dim_generos.count()} linhas.")

In [0]:
df_dim_filmes = spark.table("workspace.gold.dim_filmes").select("sk_filme", "id_filme")
df_dim_generos = spark.table("workspace.gold.dim_generos")

df_gold_bridge_filme_genero = df_silver_generos \
    .join(df_dim_filmes, on="id_filme", how="inner") \
    .join(df_dim_generos, on="nome_genero", how="inner") \
    .select("sk_filme", "sk_genero") \
    .distinct()

df_gold_bridge_filme_genero.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.bridge_filme_genero")

print(f"Tabela gold.bridge_filme_genero gravada com {df_gold_bridge_filme_genero.count()} linhas.")

In [0]:
# Célula 1: Dimensão de Pessoas (Ator, Diretor, Roteirista)
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_silver_pessoas_empresas = spark.table("workspace.silver.tb_pessoas_empresas")

df_pessoas_distintas = df_silver_pessoas_empresas \
    .filter(F.col("tipo_entidade") != "Produtora") \
    .select("nome_pessoa_empresa").distinct() \
    .withColumnRenamed("nome_pessoa_empresa", "nome_pessoa")

window_spec_pessoa = Window.orderBy("nome_pessoa")

df_gold_dim_pessoas = df_pessoas_distintas \
    .withColumn("sk_pessoa", F.row_number().over(window_spec_pessoa)) \
    .select("sk_pessoa", "nome_pessoa")

df_gold_dim_pessoas.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.dim_pessoas")

print(f"Tabela gold.dim_pessoas gravada com {df_gold_dim_pessoas.count()} linhas.")

In [0]:
# Célula 2: Dimensão de Empresas (Produtora)
df_empresas_distintas = df_silver_pessoas_empresas \
    .filter(F.col("tipo_entidade") == "Produtora") \
    .select("nome_pessoa_empresa").distinct() \
    .withColumnRenamed("nome_pessoa_empresa", "nome_empresa")

window_spec_empresa = Window.orderBy("nome_empresa")

df_gold_dim_empresas = df_empresas_distintas \
    .withColumn("sk_empresa", F.row_number().over(window_spec_empresa)) \
    .select("sk_empresa", "nome_empresa")

df_gold_dim_empresas.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.dim_empresas")

print(f"Tabela gold.dim_empresas gravada com {df_gold_dim_empresas.count()} linhas.")

In [0]:
# Célula 1: Bridge Filme-Pessoa (com o papel como atributo da participação)
df_dim_filmes = spark.table("workspace.gold.dim_filmes").select("sk_filme", "id_filme")
df_dim_pessoas = spark.table("workspace.gold.dim_pessoas")
df_dim_empresas = spark.table("workspace.gold.dim_empresas")

df_gold_bridge_filme_pessoa = df_silver_pessoas_empresas \
    .filter(F.col("tipo_entidade") != "Produtora") \
    .join(df_dim_filmes, on="id_filme", how="inner") \
    .join(df_dim_pessoas, df_silver_pessoas_empresas["nome_pessoa_empresa"] == df_dim_pessoas["nome_pessoa"], how="inner") \
    .select("sk_filme", "sk_pessoa", F.col("tipo_entidade").alias("tipo_participacao")) \
    .distinct()

df_gold_bridge_filme_pessoa.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.bridge_filme_pessoa")

print(f"Tabela gold.bridge_filme_pessoa gravada com {df_gold_bridge_filme_pessoa.count()} linhas.")

In [0]:
# Célula 2: Bridge Filme-Empresa
df_gold_bridge_filme_empresa = df_silver_pessoas_empresas \
    .filter(F.col("tipo_entidade") == "Produtora") \
    .join(df_dim_filmes, on="id_filme", how="inner") \
    .join(df_dim_empresas, df_silver_pessoas_empresas["nome_pessoa_empresa"] == df_dim_empresas["nome_empresa"], how="inner") \
    .select("sk_filme", "sk_empresa") \
    .distinct()

df_gold_bridge_filme_empresa.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.bridge_filme_empresa")

print(f"Tabela gold.bridge_filme_empresa gravada com {df_gold_bridge_filme_empresa.count()} linhas.")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_silver_avaliacoes = spark.table("workspace.silver.tb_avaliacoes_usuarios")
df_dim_filmes = spark.table("workspace.gold.dim_filmes").select("sk_filme", "id_filme")

window_spec_avaliacao = Window.orderBy("sk_filme", "nome_usuario")

df_gold_fact_avaliacoes = df_silver_avaliacoes \
    .join(df_dim_filmes, on="id_filme", how="inner") \
    .select("sk_filme", "nome_usuario", "nota_usuario", "comentario_usuario") \
    .withColumn("sk_avaliacao", F.row_number().over(window_spec_avaliacao)) \
    .select("sk_avaliacao", "sk_filme", "nome_usuario", "nota_usuario", "comentario_usuario")

df_gold_fact_avaliacoes.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.fact_avaliacoes")

print(f"Tabela gold.fact_avaliacoes gravada com {df_gold_fact_avaliacoes.count()} linhas.")

In [0]:
df_dim_filmes = spark.table("workspace.gold.dim_filmes").select("sk_filme", "id_filme")
df_silver_financeiro = spark.table("workspace.silver.tb_financeiro_filmes")
df_silver_metricas = spark.table("workspace.silver.tb_metricas_engajamento")

df_gold_fact_desempenho = df_dim_filmes \
    .join(df_silver_financeiro, on="id_filme", how="left") \
    .join(df_silver_metricas, on="id_filme", how="left") \
    .select(
        "sk_filme",
        "orcamento_usd", "receita_usd", "orcamento_brl", "receita_brl",
        "lucro_usd", "lucro_brl", "margem_lucro_percentual",
        "popularidade", "nota_media_tmdb", "qtd_votos_tmdb",
        "nota_media_imdb", "qtd_votos_imdb"
    )

df_gold_fact_desempenho.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.fact_desempenho_filmes")

print(f"Tabela gold.fact_desempenho_filmes gravada com {df_gold_fact_desempenho.count()} linhas.")

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.dim_movies AS
SELECT
  sk_filme AS sk_movie_id,
  id_filme,
  titulo,
  data_lancamento,
  ano_lancamento,
  duracao_minutos,
  idioma_original,
  status_filme,
  sinopse,
  titulo_original,
  frase_divulgacao
FROM workspace.gold.dim_filmes;

CREATE OR REPLACE TABLE workspace.gold.dim_genres AS
SELECT sk_genero AS sk_genre_id, nome_genero
FROM workspace.gold.dim_generos;

CREATE OR REPLACE TABLE workspace.gold.dim_companies AS
SELECT sk_empresa AS sk_company_id, nome_empresa AS nome_produtora
FROM workspace.gold.dim_empresas;

CREATE OR REPLACE TABLE workspace.gold.bridge_movie_genre AS
SELECT sk_filme AS sk_movie_id, sk_genero AS sk_genre_id
FROM workspace.gold.bridge_filme_genero;

CREATE OR REPLACE TABLE workspace.gold.bridge_movie_company AS
SELECT sk_filme AS sk_movie_id, sk_empresa AS sk_company_id
FROM workspace.gold.bridge_filme_empresa;

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_silver_pessoas_empresas = spark.table("workspace.silver.tb_pessoas_empresas")

df_pessoas_tipo_distintas = df_silver_pessoas_empresas \
    .filter(F.col("tipo_entidade") != "Produtora") \
    .select(
        F.col("nome_pessoa_empresa").alias("nome_pessoa"),
        F.col("tipo_entidade").alias("tipo_pessoa")
    ) \
    .distinct()

window_spec_pessoa = Window.orderBy("nome_pessoa", "tipo_pessoa")

df_gold_dim_people = df_pessoas_tipo_distintas \
    .withColumn("sk_person_id", F.row_number().over(window_spec_pessoa)) \
    .select("sk_person_id", "nome_pessoa", "tipo_pessoa")

df_gold_dim_people.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.dim_people")

print(f"Tabela gold.dim_people gravada com {df_gold_dim_people.count()} linhas.")

In [0]:
df_dim_movies = spark.table("workspace.gold.dim_movies").select("sk_movie_id", "id_filme")
df_dim_people = spark.table("workspace.gold.dim_people")

df_gold_bridge_movie_person = df_silver_pessoas_empresas \
    .filter(F.col("tipo_entidade") != "Produtora") \
    .join(df_dim_movies, on="id_filme", how="inner") \
    .join(
        df_dim_people,
        (df_silver_pessoas_empresas["nome_pessoa_empresa"] == df_dim_people["nome_pessoa"]) &
        (df_silver_pessoas_empresas["tipo_entidade"] == df_dim_people["tipo_pessoa"]),
        how="inner"
    ) \
    .select("sk_movie_id", "sk_person_id") \
    .distinct()

df_gold_bridge_movie_person.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.bridge_movie_person")

print(f"Tabela gold.bridge_movie_person gravada com {df_gold_bridge_movie_person.count()} linhas.")

In [0]:
from pyspark.sql import functions as F

df_dim_movies = spark.table("workspace.gold.dim_movies").select("sk_movie_id", "id_filme")
df_silver_financeiro = spark.table("workspace.silver.tb_financeiro_filmes")
df_silver_metricas = spark.table("workspace.silver.tb_metricas_engajamento")

df_gold_fact_movies_performance = df_dim_movies \
    .join(df_silver_financeiro, on="id_filme", how="left") \
    .join(df_silver_metricas, on="id_filme", how="left") \
    .select(
        "sk_movie_id",
        F.col("orcamento_usd").try_cast("decimal(18,2)").alias("orcamento_usd"),
        F.col("receita_usd").try_cast("decimal(18,2)").alias("receita_usd"),
        F.col("lucro_usd").try_cast("decimal(18,2)").alias("lucro_usd"),
        F.col("orcamento_brl").try_cast("decimal(18,2)").alias("orcamento_brl"),
        F.col("receita_brl").try_cast("decimal(18,2)").alias("receita_brl"),
        F.col("lucro_brl").try_cast("decimal(18,2)").alias("lucro_brl"),
        "popularidade",
        "nota_media_tmdb",
        "qtd_votos_tmdb",
        "nota_media_imdb",
        "qtd_votos_imdb"
    )

df_gold_fact_movies_performance.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.fact_movies_performance")

print(f"Tabela gold.fact_movies_performance gravada com {df_gold_fact_movies_performance.count()} linhas.")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_silver_avaliacoes = spark.table("workspace.silver.tb_avaliacoes_usuarios")
df_dim_movies = spark.table("workspace.gold.dim_movies").select("sk_movie_id", "id_filme")

df_avaliacoes_agregadas = df_silver_avaliacoes \
    .groupBy("id_filme") \
    .agg(
        F.count(F.lit(1)).alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"), 2).alias("nota_media_usuarios")
    )

df_avaliacoes_com_sk = df_avaliacoes_agregadas.join(df_dim_movies, on="id_filme", how="inner")

window_spec_review = Window.orderBy("sk_movie_id")

df_gold_dim_reviews = df_avaliacoes_com_sk \
    .withColumn("sk_review_id", F.row_number().over(window_spec_review)) \
    .select("sk_review_id", "sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios")

df_gold_dim_reviews.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.dim_reviews")

print(f"Tabela gold.dim_reviews gravada com {df_gold_dim_reviews.count()} linhas.")

In [0]:
tabelas_antigas = [
    "dim_filmes", "dim_generos", "dim_pessoas", "dim_empresas",
    "fact_desempenho_filmes", "fact_avaliacoes",
    "bridge_filme_genero", "bridge_filme_pessoa", "bridge_filme_empresa"
]

for tabela in tabelas_antigas:
    spark.sql(f"DROP TABLE IF EXISTS workspace.gold.{tabela}")
    print(f"Removida: workspace.gold.{tabela}")

In [0]:
from pyspark.sql import functions as F

df_dim_movies = spark.table("workspace.gold.dim_movies")
df_fact_performance = spark.table("workspace.gold.fact_movies_performance")
df_bridge_person = spark.table("workspace.gold.bridge_movie_person")
df_dim_people = spark.table("workspace.gold.dim_people")

# Junta a ponte com a dimensão de pessoas para saber nome e papel de cada participação
df_pessoas_por_filme = df_bridge_person.join(df_dim_people, on="sk_person_id", how="inner")

# Atores principais: pega até 3 nomes por filme (a origem não preserva ordem de relevância do elenco)
df_atores_agregados = df_pessoas_por_filme \
    .filter(F.col("tipo_pessoa") == "Ator") \
    .groupBy("sk_movie_id") \
    .agg(F.array_join(F.slice(F.collect_set("nome_pessoa"), 1, 3), ", ").alias("atores_principais"))

# Diretor(es): concatena todos os diretores cadastrados (a maioria dos filmes tem só um)
df_diretores_agregados = df_pessoas_por_filme \
    .filter(F.col("tipo_pessoa") == "Diretor") \
    .groupBy("sk_movie_id") \
    .agg(F.array_join(F.collect_set("nome_pessoa"), " e ").alias("diretor"))

# Junta tudo em torno da dim_movies (left join preserva todos os 97.879 filmes)
df_contexto = df_dim_movies.select("sk_movie_id", "id_filme", "titulo", "ano_lancamento", "sinopse") \
    .join(df_fact_performance.select("sk_movie_id", "receita_usd", "orcamento_usd"), on="sk_movie_id", how="left") \
    .join(df_atores_agregados, on="sk_movie_id", how="left") \
    .join(df_diretores_agregados, on="sk_movie_id", how="left")

# Trata cada campo individualmente ANTES da concatenação final (evita a "casca de banana" dos nulos)
df_contexto_tratado = df_contexto.select(
    "id_filme",
    "titulo",
    F.coalesce(F.col("ano_lancamento").cast("string"), F.lit("ano não informado")).alias("ano_fmt"),
    F.coalesce(F.concat(F.lit("US$ "), F.format_number(F.col("receita_usd"), 2)), F.lit("receita não divulgada")).alias("receita_fmt"),
    F.coalesce(F.concat(F.lit("US$ "), F.format_number(F.col("orcamento_usd"), 2)), F.lit("orçamento não divulgado")).alias("orcamento_fmt"),
    F.coalesce(F.col("atores_principais"), F.lit("elenco não informado")).alias("atores_fmt"),
    F.coalesce(F.col("diretor"), F.lit("diretor não informado")).alias("diretor_fmt"),
    F.coalesce(F.col("sinopse"), F.lit("sinopse não disponível")).alias("sinopse_fmt")
)

# Agora sim, a concatenação final é segura: nenhum campo chega NULL até aqui
df_gold_genai_context = df_contexto_tratado.select(
    F.col("id_filme").alias("movie_id"),
    F.col("titulo").alias("title"),
    F.concat(
        F.lit("O filme "), F.col("titulo"),
        F.lit(", lançado no ano de "), F.col("ano_fmt"),
        F.lit(", faturou "), F.col("receita_fmt"),
        F.lit(" e teve um custo de "), F.col("orcamento_fmt"),
        F.lit(". Estrelado por "), F.col("atores_fmt"),
        F.lit(" e dirigido por "), F.col("diretor_fmt"),
        F.lit(", o filme possui a seguinte sinopse: "), F.col("sinopse_fmt"),
        F.lit(".")
    ).alias("llm_context_document")
)

df_gold_genai_context.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.gold_genai_movies_context")

print(f"Tabela gold.gold_genai_movies_context gravada com {df_gold_genai_context.count()} linhas.")